In [ ]:
# ---- 0. Environment ---------------------------------------------------------
set.seed(1)
suppressPackageStartupMessages({
  library(dplyr)
  library(tidyr)
  library(ggplot2)
  library(gridExtra)
  library(grid)
  library(RColorBrewer)
})

In [ ]:
# ---- 1. Parameters ----------------------------------------------------------
date_str <- "2026-07-29_2"

data_dir <- "/project/work/2026-07-29_trajectory_class/Data/"
plot_dir <- "/project/work/2026-07-29_trajectory_class/Plot/"

celltype_order <- c("Rod", "Cone", "AC", "BC", "HC", "MG", "RGC", "RPC")
stage_order <- c("Early", "Mid", "Late","Young","Old")
direction_order <- c("central_up", "peripheral_up")

cols.palette2 <- c(
  brewer.pal(12, "Paired"),
  brewer.pal(8, "Set2"),
  brewer.pal(8, "Dark2"),
  brewer.pal(9, "Set1")
)

cols.tissue <- cols.palette2[9:11]
names(cols.tissue) <- c("fovea", "macula", "peripheral")

direction_colors <- c(
  "central_up" = "#6A3D9A",
  "peripheral_up" = "#E69F00"
)

# 数据准备

In [ ]:
# ---- 2. Read Data -----------------------------------------------------------

dev_df <- read.csv("/project/work/2026-07-06_development_DEG/Data/2026-07-06_2_regional_degs_filtered_merged.csv", stringsAsFactors = FALSE)

# 检查是否存在必要的列（可注释掉）
required_cols <- c("celltype", "stage", "gene", "direction")
missing_cols <- setdiff(required_cols, colnames(dev_df))
if (length(missing_cols) > 0) {
  stop("Missing required columns: ", paste(missing_cols, collapse = ", "))
}
head(dev_df)

In [ ]:
dev_df$direction_raw <- dev_df$direction
head(dev_df,2)

In [ ]:
dev_df$direction[dev_df$direction == "macula_up"] <- "central_up"

In [ ]:
write.csv(dev_df, "/project/work/2026-07-06_development_DEG/Data/2026-07-06_2_regional_degs_filtered_merged.csv", row.names = FALSE)

In [ ]:
adu_df <- read.csv("/project/work/2026-06-10_adult_DEG/Data/02_regional_degs/2026-07-28_adult_regional_degs_filtered_merged.csv", stringsAsFactors = FALSE)

# 检查是否存在必要的列（可注释掉）
required_cols <- c("celltype", "stage", "gene", "direction")
missing_cols <- setdiff(required_cols, colnames(adu_df))
if (length(missing_cols) > 0) {
  stop("Missing required columns: ", paste(missing_cols, collapse = ", "))
}
head(adu_df)

In [ ]:
adu_df$direction_raw <- adu_df$direction
head(adu_df,2)

In [ ]:
adu_df$direction[adu_df$direction == "Central_up"] <- "central_up"

In [ ]:
write.csv(adu_df, "/project/work/2026-06-10_adult_DEG/Data/02_regional_degs/2026-07-28_adult_regional_degs_filtered_merged.csv", row.names = FALSE)

# 读取数据

In [ ]:
dev_df <- read.csv("/project/work/2026-07-06_development_DEG/Data/2026-07-06_2_regional_degs_filtered_merged.csv", stringsAsFactors = FALSE)
head(dev_df,2)
adu_df <- read.csv("/project/work/2026-06-10_adult_DEG/Data/02_regional_degs/2026-07-28_adult_regional_degs_filtered_merged.csv", stringsAsFactors = FALSE)
head(adu_df,2)

In [ ]:
regional_df <- rbind(adu_df, dev_df)
head(regional_df,2)

# barplot

In [ ]:
# ---- 3. Prepare Count Table -------------------------------------------------
count_df <- regional_df %>%
  distinct(celltype, stage, direction, gene) %>%
  count(celltype, stage, direction, name = "n") %>%
  filter(direction %in% direction_order) %>%
  mutate(
    stage = factor(stage, levels = stage_order),
    celltype = factor(celltype, levels = celltype_order),
    direction = factor(direction, levels = direction_order),
    direction_label = ifelse(
      direction == "central_up",
      "central_up",
      "peripheral_up"
    ),
    direction_label = factor(
      direction_label,
      levels = c("central_up", "peripheral_up")
    ),
    tissue_label = ifelse(direction == "central_up", "central", "peripheral")
  ) %>%
  arrange(stage, celltype, direction)
head(count_df)

In [ ]:
total_df <- count_df %>%
  group_by(stage, celltype) %>%
  summarise(total = sum(n), .groups = "drop")
head(total_df)

In [ ]:
# ---- 4. Diverging Barplot ---------------------------------------------------
diverging_df <- count_df %>%
  mutate(plot_n = ifelse(direction == "peripheral_up", -n, n))

p_diverging <- ggplot(diverging_df, aes(x = celltype, y = plot_n, fill = direction_label)) +
  geom_hline(yintercept = 0, color = "gray25", linewidth = 0.35) +
  geom_col(width = 0.72, color = "gray35", linewidth = 0.2) +
  geom_text(
    aes(label = n, vjust = ifelse(plot_n >= 0, -0.35, 1.25)),
    color = "black", fontface = "bold", size = 3
  ) +
  facet_grid(. ~ stage, scales = "free_x", space = "free_x") +
  scale_fill_manual(values = direction_colors, name = "Direction") +
  scale_y_continuous(
    labels = function(x) abs(x),
    expand = expansion(mult = c(0.15, 0.15))
  ) +
  labs(
    title = "Regional DEG Counts by Stage and Cell Type",
    subtitle = "central_up above zero; Peripheral-up below zero (padj < 0.05, |log2FC| > 0.5)",
    x = "Cell Type",
    y = "Number of DEGs"
  ) +
  coord_cartesian(clip = "off") +
  theme_bw(base_size = 12) +
  theme(
    axis.text.x = element_text(angle = 45, hjust = 1, size = 12, color = "black"),
    axis.text.y = element_text(size = 10),
    panel.grid.major.x = element_blank(),
    panel.grid.minor = element_blank(),
    strip.background = element_rect(fill = "gray90", color = "gray70"),
    strip.text = element_text(face = "bold", size = 11),
    legend.position = "right",
    plot.title = element_text(face = "bold", hjust = 0.5),
    plot.subtitle = element_text(hjust = 0.5, size = 10)
  )
options(repr.plot.width = 24, repr.plot.height = 8)
p_diverging

In [ ]:
ggsave(
  file.path(plot_dir, paste0(date_str, "_regional_DEG_count_barplot_1.pdf")),
  p_diverging, width = 24, height = 8, dpi = 300, bg = "white"
)
# ggsave(
#   file.path(plot_dir, paste0(date_str, "_regional_DEG_count_diverging_barplot.png")),
#   p_diverging, width = 24, height = 8, dpi = 300, bg = "white"
# )